# Silver: [ENTITY_NAME]

**Layer:** Silver | **Source:** `raw/[source_path]/*.csv` | **Target:** `silver.[TABLE_NAME]`  
**Owner:** [owner_email]

## Pipeline Steps
1. Load raw CSVs from ADLS
2. Select, rename, and normalize source filename
3. Cast types, filter invalid rows
4. Deduplicate by ([DEDUP_KEYS])
5. MERGE into Delta table
6. Cleanup raw files

## Parameters
| Widget | Default | Description |
|--------|---------|-------------|
| `raw_container` | `raw` | ADLS container for raw data |
| `silver_container` | `silver` | ADLS container for silver tables |

## Customization Checklist
- [ ] Update header metadata above
- [ ] Create a `pipeline_helpers_[entity].py` in utils/ using FieldConfig (see pipeline_helpers.py)
- [ ] Update the raw file path in Step 1
- [ ] Update the Delta table DDL in Step 5
- [ ] Update the MERGE keys in Step 5
- [ ] Update the dedup window keys in Step 4

In [ ]:
import sys
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType
from pyspark.sql.functions import input_file_name, col, lower, row_number, expr
from pyspark.sql.window import Window
from functools import reduce

_utils_dir = "/Workspace/Repos/christopher.furu@hotmail.com/DataEngineering/databricks/real_estate/notebooks/utils"
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

# TODO: Replace with your entity-specific helpers module
import pipeline_helpers
from pipeline_helpers import log_step

In [ ]:
%run ../utils/pyutils

In [ ]:
%run ../utils/udf_helpers

# Configuration

In [ ]:
dbutils.widgets.text("raw_container", "raw", "Raw container name")
dbutils.widgets.text("silver_container", "silver", "Silver container name")

raw_container = dbutils.widgets.get("raw_container")
silver_container = dbutils.widgets.get("silver_container")
ds_raw = get_wasbs_path(container=raw_container)
ds_silver = get_wasbs_path(container=silver_container)

TYPE_MAP = {
    "IntegerType": IntegerType,
    "StringType": StringType,
    "DoubleType": DoubleType,
    "FloatType": FloatType,
}
# TODO: Replace SOLD_SCHEMA_FIELDS with your entity's schema fields
sold_schema = StructType([
    StructField(name, TYPE_MAP[type_name](), True)
    for name, type_name in pipeline_helpers.SOLD_SCHEMA_FIELDS
])

log_step("CONFIG", f"raw={ds_raw}  silver={ds_silver}")

# Step 1: Load Raw Data

In [ ]:
# TODO: Update the raw file path for your entity
files = [f.path for f in dbutils.fs.ls(f"{ds_raw}/[SOURCE_PATH]/") if f.path.endswith(".csv")]

if not files:
    log_step("LOAD", "No raw CSV files found. Exiting notebook.")
    dbutils.notebook.exit("NO_DATA")

dfs = [
    spark.read
         .format("csv")
         .option("header", True)
         .option("inferSchema", False)
         .schema(sold_schema)
         .load(path)
         .withColumn("source_file", input_file_name())
    for path in files
]
df_all = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)

log_step("LOAD", f"Loaded {df_all.count()} rows from {len(files)} files")

# Step 2: Select, Rename, and Normalize

In [ ]:
df_all = df_all.select(*pipeline_helpers.SELECT_COLUMNS)
df_all = pipeline_helpers.rename_columns(df_all)
df_all = (
    df_all
    .withColumn("sourceFileName", expr("normalize_source_filename(source_file)"))
    .drop("source_file")
)
log_step("SELECT_RENAME", f"Columns: {df_all.columns}")

# Step 3: Cast Types and Filter

In [ ]:
for column, dtype in pipeline_helpers.CAST_COLUMN_TYPES.items():
    df_all = df_all.withColumn(column, col(column).cast(dtype))

# TODO: Add entity-specific filters
# df_all = df_all.filter(~lower(col("url")).like("%annons%"))

for required in pipeline_helpers.REQUIRED_NON_NULL_COLUMNS:
    df_all = df_all.filter(col(required).isNotNull())

log_step("CAST_FILTER", f"{df_all.count()} rows after filtering")

# Step 4: Deduplicate

In [ ]:
# TODO: Update partition and order keys for your entity
window_spec = Window.partitionBy("[KEY1]", "[KEY2]").orderBy(col("[ORDER_COL]").desc())
df_clean = (
    df_all
    .withColumn("row_rank", row_number().over(window_spec))
    .filter(col("row_rank") == 1)
    .drop("row_rank")
)
df_clean.createOrReplaceTempView("[ENTITY]_deduplicated")
log_step("DEDUP", f"{df_clean.count()} rows after deduplication")

# Step 5: Merge into Silver

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# TODO: Update table name, schema, and location
create_table_query = f"""
CREATE TABLE IF NOT EXISTS silver.[TABLE_NAME]
(
  -- TODO: Define your column schema here
  id INT,
  name STRING
)
USING DELTA
LOCATION '{ds_silver}/[entity_path]'
"""
spark.sql(create_table_query)
log_step("TABLE", "silver.[TABLE_NAME] ensured")

In [ ]:
# TODO: Update table name and merge keys
try:
    spark.sql("""
        MERGE INTO silver.[TABLE_NAME] AS T
        USING [ENTITY]_deduplicated AS S
        ON T.[KEY1] = S.[KEY1] AND T.[KEY2] = S.[KEY2]
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    log_step("MERGE", "Completed successfully")
except Exception as e:
    log_step("MERGE_FAILED", str(e))
    raise

# Step 6: Validate

In [ ]:
row_count = spark.sql("SELECT COUNT(*) AS cnt FROM silver.[TABLE_NAME]").collect()[0]["cnt"]
log_step("VALIDATE", f"silver.[TABLE_NAME] total rows: {row_count}")
spark.sql("DESCRIBE HISTORY silver.[TABLE_NAME]").show(5, truncate=False)

# Step 7: Cleanup Raw Files

In [ ]:
try:
    if len(files) > 0:
        dbutils.fs.rm(f"{ds_raw}/[SOURCE_PATH]/", recurse=True)
        log_step("CLEANUP", f"Removed raw files")
    else:
        log_step("CLEANUP", "Skipped: no raw files processed")
except Exception as e:
    log_step("CLEANUP_FAILED", str(e))
    # Don't re-raise: data is already merged, cleanup can be retried manually